# 🥇 Module 3: Gold Layer - Dimensional Model

## Overview

In this notebook, we'll transform the Silver data into a dimensional model (Star Schema) optimized for analytics.

**What you'll learn:**
- Create dimension tables (Date, Time, Station)
- Create fact table with foreign keys
- Implement surrogate keys
- Understand Slowly Changing Dimensions (SCD) and when to use Type 1 or Type 2
- Build analytical queries

---

**Prerequisites:**
- Completed Module 2 (Silver Layer)
- `silver_trips` table exists with valid data

# 🥇 Module 3: Gold Layer - Dimensional Model

## Overview

In this notebook, we'll transform the Silver data into a dimensional model (Star Schema) optimized for analytics.

**What you'll learn:**
- Create dimension tables (Date, Time, Station)
- Create fact table with foreign keys
- Implement surrogate keys
- Understand Slowly Changing Dimensions (SCD) and when to use Type 1 or Type 2
- Build analytical queries

---

**Prerequisites:**
- Completed Module 2 (Silver Layer)
- `silver_trips` table exists with valid data

## Step 1: Verify Silver Data is Ready

First, let's make sure our Silver table has the data we need.

In [ ]:
%%sql
-- Check Silver table is ready
SELECT 
    COUNT(*) as total_records,
    SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) as valid_records,
    MIN(started_at) as min_date,
    MAX(started_at) as max_date,
    COUNT(DISTINCT start_station_id) as unique_start_stations,
    COUNT(DISTINCT end_station_id) as unique_end_stations
FROM silver_trips
WHERE is_valid = TRUE

In [ ]:
%%sql
-- Create the separate Gold schema
CREATE SCHEMA IF NOT EXISTS gold

## Step 2: Create DimDate (Date Dimension)

The Date dimension contains calendar attributes that enable time-based analysis.

**Key Design Decisions:**
- Surrogate key format: YYYYMMDD (e.g., 20260115)
- Generates all dates in the data range
- Includes business-relevant attributes

In [ ]:
%%sql
-- Create Date Dimension
CREATE OR REPLACE TABLE gold.dim_date
USING DELTA
AS
WITH date_range AS (
    -- Get the date range from our data
    SELECT 
        DATE(MIN(started_at)) as start_date,
        DATE(MAX(started_at)) as end_date
    FROM silver_trips
    WHERE is_valid = TRUE
),
date_sequence AS (
    -- Generate all dates in the range
    SELECT 
        explode(
            sequence(
                (SELECT start_date FROM date_range),
                (SELECT end_date FROM date_range),
                interval 1 day
            )
        ) as full_date
)
SELECT 
    -- Surrogate key: YYYYMMDD format
    CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT) as date_key,
    
    -- The actual date
    full_date,
    
    -- Year attributes
    YEAR(full_date) as year,
    QUARTER(full_date) as quarter,
    
    -- Month attributes
    MONTH(full_date) as month,
    DATE_FORMAT(full_date, 'MMMM') as month_name,
    DATE_FORMAT(full_date, 'MMM') as month_short,
    
    -- Week attributes
    WEEKOFYEAR(full_date) as week_of_year,
    
    -- Day attributes
    DAY(full_date) as day_of_month,
    DAYOFWEEK(full_date) as day_of_week,  -- 1=Sunday, 7=Saturday
    DATE_FORMAT(full_date, 'EEEE') as day_name,
    DATE_FORMAT(full_date, 'EEE') as day_short,
    DAYOFYEAR(full_date) as day_of_year,
    
    -- Weekend flag (Saturday=7, Sunday=1 in Spark)
    CASE 
        WHEN DAYOFWEEK(full_date) IN (1, 7) THEN TRUE 
        ELSE FALSE 
    END as is_weekend,
    
    -- Useful for sorting
    DATE_FORMAT(full_date, 'yyyy-MM') as year_month,
    DATE_FORMAT(full_date, 'yyyy-Qq') as year_quarter
    
FROM date_sequence
ORDER BY full_date

In [ ]:
%%sql
-- Verify DimDate
SELECT * FROM gold.dim_date LIMIT 10

In [ ]:
%%sql
-- Check DimDate statistics
SELECT 
    COUNT(*) as total_days,
    MIN(full_date) as first_date,
    MAX(full_date) as last_date,
    SUM(CASE WHEN is_weekend THEN 1 ELSE 0 END) as weekend_days,
    SUM(CASE WHEN NOT is_weekend THEN 1 ELSE 0 END) as weekdays
FROM gold.dim_date

## Step 3: Create DimTime (Time Dimension)

The Time dimension contains time-of-day attributes for analyzing patterns within each day.

**Key Design Decisions:**
- Surrogate key format: HHMM (e.g., 0830 for 8:30 AM)
- One row per minute of the day (1,440 rows)
- Includes rush hour flags for transportation analysis

In [ ]:
%%sql
-- Create Time Dimension
CREATE OR REPLACE TABLE gold.dim_time
USING DELTA
AS
WITH hours AS (
    SELECT explode(sequence(0, 23, 1)) as hour
),
minutes AS (
    SELECT explode(sequence(0, 59, 1)) as minute
),
time_values AS (
    SELECT 
        h.hour,
        m.minute
    FROM hours h
    CROSS JOIN minutes m
)
SELECT 
    -- Surrogate key: HHMM format (e.g., 830 for 08:30)
    (hour * 100 + minute) as time_key,
    
    -- Hour and minute
    hour,
    minute,
    
    -- Formatted time string
    LPAD(CAST(hour AS STRING), 2, '0') || ':' || LPAD(CAST(minute AS STRING), 2, '0') as time_string,
    
    -- 12-hour format
    CASE WHEN hour = 0 THEN 12
         WHEN hour > 12 THEN hour - 12
         ELSE hour 
    END as hour_12,
    CASE WHEN hour < 12 THEN 'AM' ELSE 'PM' END as am_pm,
    
    -- Time of day classification
    CASE 
        WHEN hour >= 5 AND hour < 9 THEN 'Early Morning'
        WHEN hour >= 9 AND hour < 12 THEN 'Morning'
        WHEN hour >= 12 AND hour < 14 THEN 'Midday'
        WHEN hour >= 14 AND hour < 17 THEN 'Afternoon'
        WHEN hour >= 17 AND hour < 20 THEN 'Evening'
        WHEN hour >= 20 AND hour < 23 THEN 'Night'
        ELSE 'Late Night'
    END as time_of_day,
    
    -- Rush hour flag (7-9 AM and 4-7 PM)
    CASE 
        WHEN (hour >= 7 AND hour < 9) OR (hour >= 16 AND hour < 19) THEN TRUE
        ELSE FALSE
    END as is_rush_hour,
    
    -- Morning/Afternoon rush
    CASE 
        WHEN hour >= 7 AND hour < 9 THEN 'Morning Rush'
        WHEN hour >= 16 AND hour < 19 THEN 'Evening Rush'
        ELSE 'Off-Peak'
    END as rush_period,
    
    -- Useful for grouping
    FLOOR(hour / 6) as quarter_of_day  -- 0=Night, 1=Morning, 2=Afternoon, 3=Evening
    
FROM time_values
ORDER BY hour, minute

In [ ]:
%%sql
-- Verify DimTime
SELECT * FROM gold.dim_time WHERE hour = 8 LIMIT 10

In [ ]:
%%sql
-- Check DimTime statistics
SELECT 
    COUNT(*) as total_time_slots,
    SUM(CASE WHEN is_rush_hour THEN 1 ELSE 0 END) as rush_hour_slots,
    COUNT(DISTINCT time_of_day) as time_periods
FROM gold.dim_time

## Step 4: Create Start and End Station Dimensions

The station dimensions contain station attributes for each trip role. Start and end stations are kept in separate tables so each role has its own surrogate-key namespace.

**Key Design Decisions:**
- `gold.dim_start_station` contains stations used as trip origins
- `gold.dim_end_station` contains stations used as trip destinations
- Each table has its own surrogate key
- Both tables preserve geographic coordinates for mapping

In [ ]:
%%sql
-- Create Start Station Dimension
CREATE OR REPLACE TABLE gold.dim_start_station
USING DELTA
AS
WITH start_stations AS (
    SELECT DISTINCT
        start_station_id as station_id,
        start_station_name as station_name,
        start_station_desc as station_description,
        start_latitude as latitude,
        start_longitude as longitude
    FROM silver_trips
    WHERE is_valid = TRUE
      AND start_station_id IS NOT NULL
),
deduped_stations AS (
    SELECT 
        station_id,
        FIRST(station_name) as station_name,
        FIRST(station_description) as station_description,
        AVG(latitude) as latitude,
        AVG(longitude) as longitude
    FROM start_stations
    GROUP BY station_id
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY station_id) as station_key,
    station_id,
    station_name,
    station_description,
    ROUND(latitude, 6) as latitude,
    ROUND(longitude, 6) as longitude,
    CASE 
        WHEN latitude > 59.92 AND longitude < 10.75 THEN 'North-West'
        WHEN latitude > 59.92 AND longitude >= 10.75 THEN 'North-East'
        WHEN latitude <= 59.92 AND longitude < 10.75 THEN 'South-West'
        ELSE 'South-East'
    END as city_quadrant
FROM deduped_stations
ORDER BY station_id

In [ ]:
%%sql
-- Create End Station Dimension
CREATE OR REPLACE TABLE gold.dim_end_station
USING DELTA
AS
WITH end_stations AS (
    SELECT DISTINCT
        end_station_id as station_id,
        end_station_name as station_name,
        end_station_desc as station_description,
        end_latitude as latitude,
        end_longitude as longitude
    FROM silver_trips
    WHERE is_valid = TRUE
      AND end_station_id IS NOT NULL
),
deduped_stations AS (
    SELECT 
        station_id,
        FIRST(station_name) as station_name,
        FIRST(station_description) as station_description,
        AVG(latitude) as latitude,
        AVG(longitude) as longitude
    FROM end_stations
    GROUP BY station_id
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY station_id) as station_key,
    station_id,
    station_name,
    station_description,
    ROUND(latitude, 6) as latitude,
    ROUND(longitude, 6) as longitude,
    CASE 
        WHEN latitude > 59.92 AND longitude < 10.75 THEN 'North-West'
        WHEN latitude > 59.92 AND longitude >= 10.75 THEN 'North-East'
        WHEN latitude <= 59.92 AND longitude < 10.75 THEN 'South-West'
        ELSE 'South-East'
    END as city_quadrant
FROM deduped_stations
ORDER BY station_id

In [ ]:
%%sql
-- Verify End Station Dimension
SELECT * FROM gold.dim_end_station LIMIT 15

In [ ]:
%%sql
-- Verify Start Station Dimension
SELECT * FROM gold.dim_start_station LIMIT 15

In [ ]:
%%sql
-- Check Start Station statistics
SELECT 
    COUNT(*) as total_start_stations,
    COUNT(DISTINCT city_quadrant) as quadrants,
    MIN(latitude) as min_lat,
    MAX(latitude) as max_lat,
    MIN(longitude) as min_lon,
    MAX(longitude) as max_lon
FROM gold.dim_start_station

In [ ]:
%%sql
-- Check End Station statistics
SELECT 
    COUNT(*) as total_end_stations,
    COUNT(DISTINCT city_quadrant) as quadrants,
    MIN(latitude) as min_lat,
    MAX(latitude) as max_lat,
    MIN(longitude) as min_lon,
    MAX(longitude) as max_lon
FROM gold.dim_end_station

## Step 5: Slowly Changing Dimensions (SCD)

A dimension can change after we first load it. A bike station might be renamed, moved, or given a new description. The business key (`station_id`) stays the same, but the descriptive attributes change.

| Pattern | What happens when an attribute changes | Use it when |
|---------|------------------------------------------|-------------|
| **Type 1** | Update or replace the existing row; no history is retained. | Only the current station details matter. |
| **Type 2** | Add a new row with a new surrogate key and date range; history is retained. | Reports must show the station attributes that were true at the time of the trip. |

`dim_start_station` and `dim_end_station` in this course are **Type 1 current-state dimensions** because the notebook recreates them from the latest available source data. The following optional extension creates `gold.dim_station_scd2`, a combined **Type 2** history table for station observations.

The extension detects changes in station attributes and creates one row per version. The columns `effective_from_date`, `effective_to_date`, `is_current`, and `version_number` make the history visible.

> In production, use a reliable source-effective timestamp and incrementally `MERGE` a staging table into the Type 2 dimension. This batch exercise infers the effective date from the day the changed station attributes first appear in trip data.

After running the next cell, inspect the history with:

```sql
SELECT *
FROM gold.dim_station_scd2
ORDER BY station_id, effective_from_date;
```

A fact table that uses Type 2 keys must join each trip to the station version valid on the trip date. The core `gold.fact_trips` table below intentionally continues to use the simpler role-specific Type 1 dimensions.

In [ ]:
%%sql
-- Optional SCD Type 2 extension: preserve station attribute history.
-- This is built separately so the core fact table can continue to use dim_station.
CREATE OR REPLACE TABLE gold.dim_station_scd2
USING DELTA
AS
WITH station_observations AS (
    SELECT
        start_station_id AS station_id,
        start_station_name AS station_name,
        start_station_desc AS station_description,
        start_latitude AS latitude,
        start_longitude AS longitude,
        DATE(started_at) AS observed_date
    FROM silver_trips
    WHERE is_valid = TRUE AND start_station_id IS NOT NULL

    UNION ALL

    SELECT
        end_station_id AS station_id,
        end_station_name AS station_name,
        end_station_desc AS station_description,
        end_latitude AS latitude,
        end_longitude AS longitude,
        DATE(ended_at) AS observed_date
    FROM silver_trips
    WHERE is_valid = TRUE AND end_station_id IS NOT NULL
),
daily_station_state AS (
    -- A source can report a station many times per day. Keep one deterministic state per day.
    SELECT
        station_id,
        observed_date,
        MAX(station_name) AS station_name,
        MAX(station_description) AS station_description,
        ROUND(AVG(latitude), 6) AS latitude,
        ROUND(AVG(longitude), 6) AS longitude
    FROM station_observations
    GROUP BY station_id, observed_date
),
hashed_state AS (
    SELECT
        *,
        SHA2(CONCAT_WS('||',
            COALESCE(station_name, ''),
            COALESCE(station_description, ''),
            CAST(latitude AS STRING),
            CAST(longitude AS STRING)
        ), 256) AS attribute_hash
    FROM daily_station_state
),
change_flags AS (
    SELECT
        *,
        CASE
            WHEN LAG(attribute_hash) OVER (PARTITION BY station_id ORDER BY observed_date) IS NULL THEN 1
            WHEN attribute_hash <> LAG(attribute_hash) OVER (PARTITION BY station_id ORDER BY observed_date) THEN 1
            ELSE 0
        END AS starts_new_version
    FROM hashed_state
),
versioned_state AS (
    SELECT
        *,
        SUM(starts_new_version) OVER (
            PARTITION BY station_id
            ORDER BY observed_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS version_number
    FROM change_flags
),
version_starts AS (
    SELECT
        station_id,
        version_number,
        MIN(observed_date) AS effective_from_date,
        FIRST(station_name) AS station_name,
        FIRST(station_description) AS station_description,
        FIRST(latitude) AS latitude,
        FIRST(longitude) AS longitude
    FROM versioned_state
    GROUP BY station_id, version_number
),
version_ranges AS (
    SELECT
        *,
        LEAD(effective_from_date) OVER (
            PARTITION BY station_id
            ORDER BY effective_from_date
        ) AS next_effective_from_date
    FROM version_starts
)
SELECT
    ROW_NUMBER() OVER (ORDER BY station_id, effective_from_date) AS station_scd_key,
    station_id,
    station_name,
    station_description,
    latitude,
    longitude,
    effective_from_date,
    COALESCE(DATE_SUB(next_effective_from_date, 1), DATE('9999-12-31')) AS effective_to_date,
    next_effective_from_date IS NULL AS is_current,
    version_number
FROM version_ranges

## Step 6: Create FactTrips (Fact Table)

The Fact table is the heart of our star schema. It contains:
- Foreign keys to all dimension tables
- Measurements (duration, counts)

**The Grain: One row per individual bike trip**

In [ ]:
%%sql
-- Create Fact Table
CREATE OR REPLACE TABLE gold.fact_trips
USING DELTA
AS
SELECT 
    -- Surrogate key for the fact
    ROW_NUMBER() OVER (ORDER BY s.started_at, s.trip_id) as trip_key,
    
    -- Foreign keys to dimensions
    CAST(DATE_FORMAT(s.started_at, 'yyyyMMdd') AS INT) as date_key,
    (HOUR(s.started_at) * 100 + MINUTE(s.started_at)) as start_time_key,
    (HOUR(s.ended_at) * 100 + MINUTE(s.ended_at)) as end_time_key,
    start_station.station_key as start_station_key,
    end_station.station_key as end_station_key,
    
    -- Degenerate dimension (original trip ID for drill-through)
    s.trip_id as source_trip_id,
    
    -- Measures
    s.duration_seconds,
    ROUND(s.duration_seconds / 60.0, 2) as duration_minutes,
    1 as trip_count  -- Useful for SUM aggregations
    
FROM silver_trips s

-- Join each station role to its own dimension
INNER JOIN gold.dim_start_station start_station 
    ON s.start_station_id = start_station.station_id
INNER JOIN gold.dim_end_station end_station 
    ON s.end_station_id = end_station.station_id

WHERE s.is_valid = TRUE

In [ ]:
%%sql
-- Verify FactTrips
SELECT * FROM gold.fact_trips LIMIT 10

In [ ]:
%%sql
-- Check FactTrips statistics
SELECT 
    COUNT(*) as total_trips,
    COUNT(DISTINCT date_key) as unique_dates,
    COUNT(DISTINCT start_station_key) as unique_start_stations,
    COUNT(DISTINCT end_station_key) as unique_end_stations,
    MIN(duration_seconds) as min_duration_sec,
    MAX(duration_seconds) as max_duration_sec,
    ROUND(AVG(duration_minutes), 1) as avg_duration_min,
    SUM(trip_count) as total_trip_count
FROM gold.fact_trips

## Step 7: Validate the Star Schema

Let's verify that all foreign keys properly join to their dimension tables.

In [ ]:
%%sql
-- Validate: All date_keys in fact exist in dimension
SELECT 
    'date_key' as foreign_key,
    COUNT(DISTINCT f.date_key) as unique_keys_in_fact,
    COUNT(DISTINCT d.date_key) as matching_keys_in_dim,
    SUM(CASE WHEN d.date_key IS NULL THEN 1 ELSE 0 END) as orphan_count
FROM gold.fact_trips f
LEFT JOIN gold.dim_date d ON f.date_key = d.date_key

In [ ]:
%%sql
-- Validate: All time_keys in fact exist in dimension
SELECT 
    'start_time_key' as foreign_key,
    COUNT(DISTINCT f.start_time_key) as unique_keys_in_fact,
    SUM(CASE WHEN t.time_key IS NULL THEN 1 ELSE 0 END) as orphan_count
FROM gold.fact_trips f
LEFT JOIN gold.dim_time t ON f.start_time_key = t.time_key

UNION ALL

SELECT 
    'end_time_key' as foreign_key,
    COUNT(DISTINCT f.end_time_key) as unique_keys_in_fact,
    SUM(CASE WHEN t.time_key IS NULL THEN 1 ELSE 0 END) as orphan_count
FROM gold.fact_trips f
LEFT JOIN gold.dim_time t ON f.end_time_key = t.time_key

In [ ]:
%%sql
-- Validate: Start and end station keys exist in their role dimensions
SELECT 
    'start_station_key' as foreign_key,
    COUNT(DISTINCT f.start_station_key) as unique_keys_in_fact,
    COUNT(DISTINCT s.station_key) as matching_keys_in_dim,
    SUM(CASE WHEN s.station_key IS NULL THEN 1 ELSE 0 END) as orphan_count
FROM gold.fact_trips f
LEFT JOIN gold.dim_start_station s ON f.start_station_key = s.station_key

UNION ALL

SELECT 
    'end_station_key' as foreign_key,
    COUNT(DISTINCT f.end_station_key) as unique_keys_in_fact,
    COUNT(DISTINCT e.station_key) as matching_keys_in_dim,
    SUM(CASE WHEN e.station_key IS NULL THEN 1 ELSE 0 END) as orphan_count
FROM gold.fact_trips f
LEFT JOIN gold.dim_end_station e ON f.end_station_key = e.station_key

## Step 8: Sample Analytical Queries

Now let's demonstrate the power of our star schema with some business questions!

In [ ]:
%%sql
-- Question 1: How many trips per day of week?


In [ ]:
%%sql
-- Question 2: Rush hour vs. off-peak trips


In [ ]:
%%sql
-- Question 3: Top 10 busiest stations (start + end combined)


In [ ]:
%%sql
-- Question 4: Monthly trend analysis


In [ ]:
%%sql
-- Question 5: Weekend vs. Weekday comparison


In [ ]:
%%sql
-- Question 6: Hourly usage pattern


## Step 9: Summary - Our Complete Star Schema

In [ ]:
%%sql
-- Final model summary


## ✅ Module Complete!

### Summary

In this module, you have built a complete **Star Schema** dimensional model in the `gold` schema:

| Table | Type | Purpose |
|-------|------|--------|
| `gold.dim_date` | Dimension | Calendar attributes for time analysis |
| `gold.dim_time` | Dimension | Time-of-day attributes for intraday analysis |
| `gold.dim_start_station` | Type 1 dimension | Current origin station details |
| `gold.dim_end_station` | Type 1 dimension | Current destination station details |
| `gold.dim_station_scd2` | Optional Type 2 dimension | Historical station versions for SCD learning |
| `gold.fact_trips` | Fact | Trip measurements with all dimension keys |

### Key Takeaways

1. **Star Schema simplifies queries** - Easy joins, clear structure
2. **Role-specific dimensions clarify relationships** - Start and end stations have independent keys
3. **Surrogate keys enable flexibility** - Integer keys are fast and stable
4. **SCD Type 1 and Type 2 solve different problems** - Type 1 retains only the current state; Type 2 preserves historical versions.
5. **Facts are the measurements** - What we count, sum, and average
6. **Grain is fundamental** - One row = one trip

### What's Next?

With your dimensional model complete, you can:
- **Build Power BI reports** on top of these tables
- **Create a Semantic Model** connecting to the Gold layer
- **Add more dimensions** (weather, events, bike type)
- **Extend `gold.fact_trips`** to use the Type 2 station key through a date-range join

🎉 **Congratulations! You've completed the Data Modelling & Dimensional Models course!**